# Legacy variant: Cobalt L-edge Energy Sweep with Memory-Safe Export

This notebook is a memory-safe version of `tutorial_cobalt_l_edge_energy_sweep.ipynb`.
It keeps the original simulation model, but changes the sweep loop so the kernel
does not keep every energy result alive in RAM.

The main differences are:

- each energy is simulated, written to its own HDF5 file, then deleted;
- only a small manifest list remains in memory;
- FTH reconstructions are skipped by default because they were only used by the
  plotting cells after the sweep;
- the heavy plotting/analysis cells after the sweep are intentionally omitted.

Use this version when the original `RUN_ENERGY_SWEEP` loop crashes the kernel or
slowly grows memory until the process is killed.

The sample-to-detector selector below defaults to the fast Fraunhofer FFT. See [Tutorial 16](../16_compare_detector_propagation.ipynb) for the same-exit-wave Rayleigh–Sommerfeld comparison. 


In [ ]:
# Sample-to-detector propagation (independent of multislice).
detector_propagation_method = "fraunhofer"  # Default; opt in with "rayleigh_sommerfeld".
# Direct Rayleigh-Sommerfeld is expensive: try small grids first.


In [ ]:
import gc
import json
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import zoom

%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

from scattering_calculator.sample_generator import pattern_generator
from scattering_calculator.simulation_pipelines import simulation_configuration as sim
%matplotlib inline



## 1. User Controls

The geometry below is fixed for the whole sweep. Only `energy` changes inside
the loop.

`RUN_ENERGY_SWEEP=False` lets you inspect the notebook quickly without starting
the simulations. Set it to `True` to generate the energy-dependent arrays.

For memory safety, leave `COMPUTE_FTH_RECONSTRUCTIONS=False` unless you really
need the reconstruction arrays. The original notebook used those arrays only for
plotting after the sweep.

`farfield_oversampling` works like in `simulate_hologram_sweep.py`: values larger than 1 extend the exit wave with the physical background before the far-field FFT, so the simulated hologram is sampled more finely before detector binning.

Set `ignore_flat_detector_curvature=True` if you want the detector projection to
use a linear reciprocal-space map, `qx = k * x / z` and `qy = k * y / z`, instead
of the default flat-detector angular mapping. Leave it `False` to reproduce the
previous hologram outputs.


In [ ]:

RUN_ENERGY_SWEEP = True
COMPUTE_FTH_RECONSTRUCTIONS = False



pols=["CR","CL"]
# Energies spanning the Co L3 edge near 778 eV and the L2 edge near 793 eV.
energy_values_eV = np.arange(773.0, 803.0, 0.5)
energy_values_eV = np.array([774,775,777,777.5,778,778.5,778.9,779.5,780,780.5,784,787,788.3,789,790,791,794,800,804])

energy_values_eV = np.arange(774.0, 804.0, 2.0)
#energy_values_eV = np.array([781.25, 782, 784,786])

# Optional detector-projection energies. Leave as None to project each
# hologram with the same energy used for the refractive indices and propagation.
# Set to a scalar, e.g. np.min(energy_values_eV), to record all holograms on the
# same detector q-grid, or to an array with the same length as energy_values_eV.
detector_energy_values_eV = 774.
reference_energy_eV = 774.0#81.25

# Use Jones for the full polarization interaction. Scalar is faster and can be
# used when the selected polarization is an eigenmode of the local interaction.
propagator_method = "Scalar"
scalar_refractive_index_lazy = True

# Jones-only is fast and already includes the rank-zero longitudinal phase.
# Set propagate=True to include multislice free-space propagation between layers.
propagate = False
ignore_flat_detector_curvature = True
# True = use linear qx=k*x/z, qy=k*y/z detector projection
# Fixed sample, aperture, illumination, and magnetic texture.
recipe = "[Au(200)Cr(5)]x20/SiN(200)/Ta(7)Pt(10)Dy(42)Co(90)"
aperture_top_radius_factors = [3.,5., 5.]
if propagate ==False:
    recipe = "Au(4000)Cr(100)/SiN(200)/Ta(7)Pt(10)Dy(42)Co(90)" 
    aperture_top_radius_factors = [1.0,1.0,1.0]



jones_apply_zero_order_phase = True
multislice_propagation_roi = False#True
multislice_propagation_roi_padding_px = 64
multislice_propagation_roi_merge_overlaps = False
propagation_padding_px = 32
propagation_padding_mode = "edge"
propagation_absorber_width_px = 8
propagation_absorber_strength = 6.0
propagation_absorber_profile = "cosine"

# Detector and beam settings. The sample mesh is fixed from reference_energy_eV.
detector_shape = (1024,1024)
detector_center = tuple(np.array(detector_shape) // 2)
oversampling = 2
# Far-field oversampling extends the propagated exit wave with the
# physical background before the far-field FFT. This is independent
# of sample-grid oversampling above.
farfield_oversampling = 4
photon_flux = 0.9e11
coherence_length = (25000e-6, 25000e-6)




sample_name = "cobalt_l_edge_energy_sweep_memory_safe_nopropagation"
illumination_center = (0.0, 0.0)
illumination_focus_distance = 1e-3
illumination_fwhm = 50.e-6
illumination_alpha_beam = (0.0, 0.0)

aperture_types = ["OH", "OH", "OH"]
aperture_radii = [250e-9, 30e-9, 10e-9]
aperture_centers = [(0.0, 0.0), (-700e-9, -700e-9), (700e-9, -700e-9)]
aperture_sigmas = [4e-9, 2e-9, 2e-9]
aperture_angles = [0.0, 0.0, 0.0]
aperture_ellipticities = [1.0, 1.0, 1.0]
aperture_roughnesses = [0.0, 0.02, 0.02]
aperture_roughness_modes = [(0, 0), (3, 10), (3, 10)]
aperture_seeds = [1, 2, 3]

# The object image in an FTH reconstruction is not centered at the origin.
# It is displaced by the reference-hole/object-hole separation. Use the first
# RH by default; flip the sign if you want the conjugate/twin object image.
fth_reference_aperture_index = 1
fth_reconstruction_shift_sign = 1

# Domain averages ignore pixels close to a domain wall, where mz is neither
# clearly positive nor clearly negative after smoothing.
xmcd_domain_threshold = 0.9

pattern_type = "binary_labyrinth_pattern"
pattern_config = {
    "stripe_width": 135e-9/4,
    "sigma": 4e-9,
    "domain_conversion": "soft",
    "softness": 1.2,
    "n_steps": 60,
    "seed": 4,
    "use_gpu": False,
}


## 2. Build The Fixed Sample Mesh And Magnetic Pattern

The detector geometry and reference energy define a fixed sample-plane pixel size. The same physical magnetic pattern is reused at every energy, so changes in the result come from the energy-dependent refractive indices rather than from a different random sample.


In [ ]:
def make_xray_config(energy_eV, pol=pols[0]):
    xray_config = sim.XRayConfig(
        energy=float(energy_eV),
        photon_flux=photon_flux,
        pol=pol,
        coherence_length=coherence_length,
    )
    xray_config.setup()
    return xray_config


def detector_energy_for_index(index, energy_eV):
    """Return the detector-projection energy for one simulated energy point."""
    if detector_energy_values_eV is None:
        return float(energy_eV)
    detector_values = np.asarray(detector_energy_values_eV, dtype=float)
    if detector_values.ndim == 0:
        return float(detector_values)
    if detector_values.shape != np.asarray(energy_values_eV).shape:
        raise ValueError(
            "detector_energy_values_eV must be None, scalar, or match "
            f"energy_values_eV shape {np.asarray(energy_values_eV).shape}; "
            f"got {detector_values.shape}"
        )
    return float(detector_values[index])


def detector_energy_scale_reference():
    """Reference detector energy used for optional FTH hologram rescaling."""
    if detector_energy_values_eV is None:
        return float(np.min(energy_values_eV))
    detector_values = np.asarray(detector_energy_values_eV, dtype=float)
    if detector_values.ndim == 0:
        return float(detector_values)
    return float(np.min(detector_values))


def make_detector_config(noise_seed=20):
    beamstop_config = sim.BeamstopConfig(
        bs_method="circular",
        bs_detector_distance=0.010,
        bs_center=detector_center,
        bs_config={
            "radius": 90e-6,
            "sigma": 1e-6,
            "wire_width": 5e-6,
            "wire_bend": 30e-6,
            "angle": np.deg2rad(25),
            "antialias": 2,
            "seed": 4,
        },
    )
    detector_config = sim.DetectorConfig(
        detector_propagation_method=detector_propagation_method,
        shape=detector_shape,
        pixel_size=20e-6,
        sample_to_detector_distance=0.05674,
        detector_center=detector_center,
        detector_params={
            "readout_noise_average": 50,
            "readout_noise_sigma": 3,
            "detector_threshold": 64e3,
            "counts_per_photon": 200,
            "quantum_efficiency": 1.0,
            "noise_seed": int(noise_seed),
        },
        measurement_config={
            "number_frames": 100,
            "max_counts_per_image": None,
            "exposure_time": 1.0,
        },
        beamstop_config=beamstop_config,
        use_detector_pixel_footprint=True,
        detector_pixel_footprint_samples=oversampling,
        ignore_flat_detector_curvature=(ignore_flat_detector_curvature if detector_propagation_method == "fraunhofer" else False),
    )
    detector_config.setup()
    return detector_config


reference_xray_config = make_xray_config(reference_energy_eV)
reference_detector_config = make_detector_config()
real_space_pixel_size = reference_detector_config.calc_realspace_resolution(reference_xray_config.beam_params) / oversampling
sample_shape = np.array([0, oversampling * detector_shape[0], oversampling * detector_shape[1]], dtype=int)

magnetic_pattern_config = sim.MagneticPatternConfig(
    pattern_type_method=pattern_type,
    shape=tuple(sample_shape[1:]),
    real_space_pixel_size=real_space_pixel_size,
    pattern_config=pattern_config,
)
magnetic_pattern_config.create_pattern()
magnetic_pattern_2d = magnetic_pattern_config.magnetic_pattern

print(f"Fixed sample pixel size: {real_space_pixel_size * 1e9:.2f} nm")
print(f"Fixed lateral grid: {tuple(sample_shape[1:])}")
print(f"Energy points: {len(energy_values_eV)}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
extent_nm = np.array([-sample_shape[2] / 2, sample_shape[2] / 2, sample_shape[1] / 2, -sample_shape[1] / 2]) * real_space_pixel_size * 1e9
im = axes[0].imshow(magnetic_pattern_2d, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_nm, interpolation="nearest")
axes[0].set_title("fixed magnetic pattern mz")
axes[0].set_xlabel("x (nm)")
axes[0].set_ylabel("y (nm)")
fig.colorbar(im, ax=axes[0], label="mz")

# Draw the configured aperture centers and nominal bottom radii on top of the pattern.
axes[1].imshow(magnetic_pattern_2d, cmap="gray", extent=extent_nm, interpolation="nearest")
for typ, radius, center in zip(aperture_types, aperture_radii, aperture_centers):
    cy, cx = center
    circle = plt.Circle((cx * 1e9, cy * 1e9), radius * 1e9, fill=False, linewidth=1.5)
    axes[1].add_patch(circle)
    axes[1].plot(cx * 1e9, cy * 1e9, "+", markersize=8)
    axes[1].text(cx * 1e9, cy * 1e9, f" {typ}", va="center")
axes[1].set_title("aperture layout over fixed pattern")
axes[1].set_xlabel("x (nm)")
axes[1].set_ylabel("y (nm)")


## 3. Simulation Helpers

At each energy the notebook rebuilds the material optical constants and the
dielectric tensor, because those are energy dependent. The magnetic pattern and
aperture parameters are copied unchanged.

This memory-safe version writes the arrays needed for phase retrieval and later
analysis, but it does not keep the full simulation objects for every energy in a
Python list.


In [ ]:
def frame_average(array):
    array = np.asarray(array)
    return array.mean(axis=0) if array.ndim == 3 else array


def rescale_hologram_for_fth(hologram, scale, output_shape):
    """Enlarge a hologram spatially, then center-crop to detector shape."""
    hologram = np.asarray(hologram)
    if scale < 1:
        raise ValueError(f"FTH hologram scale must be >= 1, got {scale}.")
    zoom_factors = (1,) * (hologram.ndim - 2) + (scale, scale)
    scaled = zoom(hologram, zoom_factors, order=1, mode="nearest", prefilter=False)
    target_y, target_x = map(int, output_shape)
    start_y = (scaled.shape[-2] - target_y) // 2
    start_x = (scaled.shape[-1] - target_x) // 2
    return scaled[..., start_y:start_y + target_y, start_x:start_x + target_x]


def fth_reconstruct(hologram):
    """Reconstruct over the spatial axes without shifting a frame axis."""
    axes = (-2, -1)
    return np.fft.fftshift(
        np.fft.fft2(np.fft.fftshift(hologram, axes=axes), axes=axes),
        axes=axes,
    )


def compute_energy_aligned_reconstructions(hologram_config, scale, output_shape):
    """Reconstruct rescaled copies while preserving all native hologram stores."""
    source_stores = {
        "ideal": hologram_config.ideal_holograms,
        "detected": hologram_config.detected_holograms,
        "detected_no_beamstop": hologram_config.detected_holograms_no_beamstop,
    }
    hologram_config.reconstructions = {}
    for source, store in source_stores.items():
        hologram_config.reconstructions[source] = {
            key: fth_reconstruct(rescale_hologram_for_fth(value, scale, output_shape))
            for key, value in store.items()
        }

    hologram_config.reconstructions["exit_wave"] = {
        key: fth_reconstruct(value)
        for key, value in hologram_config.exit_waves.items()
    }


def symmetric_limits(data, percentile=99.0):
    vmax = np.nanpercentile(np.abs(data), percentile)
    return -float(vmax or 1.0), float(vmax or 1.0)


def continuous_phase(values):
    """Return a phase curve without artificial 2*pi branch-cut jumps."""
    phase = np.unwrap(np.angle(values))
    offset = 2 * np.pi * np.round(np.nanmedian(phase) / (2 * np.pi))
    return phase - offset


def continuous_complex_log_ratio(numerator, denominator, eps=1e-30):
    """Compute log(numerator / denominator) with an unwrapped imaginary part."""
    ratio = (np.asarray(numerator) + eps) / (np.asarray(denominator) + eps)
    return np.log(np.abs(ratio)) + 1j * continuous_phase(ratio)


def resample_nearest_to_shape(array, shape):
    """Nearest-neighbor resample a 2D mask/image to a requested shape."""
    array = np.asarray(array)
    if array.shape == tuple(shape):
        return array
    y_idx = np.rint(np.linspace(0, array.shape[0] - 1, shape[0])).astype(int)
    x_idx = np.rint(np.linspace(0, array.shape[1] - 1, shape[1])).astype(int)
    return array[np.ix_(y_idx, x_idx)]


def object_hole_crop_slices(shape, pixel_size, radius_factor=1.0):
    oh_index = aperture_types.index("OH")
    center_y_m, center_x_m = aperture_centers[oh_index]
    return crop_slices_around_position(shape, pixel_size, (center_y_m, center_x_m), aperture_radii[oh_index] * radius_factor)


def fth_object_image_crop_slices(shape, pixel_size, radius_factor=1.0):
    """Crop the FTH object image displaced by the selected RH-OH vector.

    In the exit wave the object hole is centered at its real-space OH position.
    In the FTH reconstruction, the object image appears in the cross-correlation
    term, shifted from the reconstruction origin by the reference-hole/object-hole
    separation. The sign selects which twin image to inspect.
    """
    oh_index = aperture_types.index("OH")
    oh_center = np.asarray(aperture_centers[oh_index], dtype=float)
    rh_center = np.asarray(aperture_centers[fth_reference_aperture_index], dtype=float)
    reconstruction_center = fth_reconstruction_shift_sign * (rh_center - oh_center)
    return crop_slices_around_position(
        shape,
        pixel_size,
        reconstruction_center,
        aperture_radii[oh_index] * radius_factor,
    )


def crop_slices_around_position(shape, pixel_size, center_m, radius_m):
    center_y_m, center_x_m = center_m
    cy = int(np.clip(round(shape[-2] / 2 + center_y_m / pixel_size), 0, shape[-2] - 1))
    cx = int(np.clip(round(shape[-1] / 2 + center_x_m / pixel_size), 0, shape[-1] - 1))
    half_width = max(4, int(np.ceil(radius_m / pixel_size)))
    y0 = max(0, cy - half_width)
    y1 = min(shape[-2], cy + half_width + 1)
    x0 = max(0, cx - half_width)
    x1 = min(shape[-1], cx + half_width + 1)
    return slice(y0, y1), slice(x0, x1)


sample_oh_slices = object_hole_crop_slices(sample_shape[1:], real_space_pixel_size, radius_factor=1.0)


def extract_material_layer_metadata(sample_config, energy_eV):
    """Record the layer optical constants used to build the propagation stack."""
    structure = sample_config.sample_structure
    layer_refractive_indices = np.asarray(
        structure.layer_refractive_indices,
        dtype=np.complex128,
    )
    effective_refractive_indices = np.asarray(
        structure.effective_refractive_indices,
        dtype=np.complex128,
    )
    return {
        "energy_eV": float(energy_eV),
        "recipe": recipe,
        "propagator_method": propagator_method,
        "scalar_refractive_index_lazy": bool(scalar_refractive_index_lazy),
        "layer_names": list(structure.layer_names),
        "layer_thicknesses_m": np.asarray(structure.layer_thicknesses, dtype=float),
        "refractive_index_channel_names": ["n_total", "n_circ", "n_lin"],
        "layer_refractive_indices": layer_refractive_indices,
        "effective_refractive_indices_m": effective_refractive_indices,
        "max_abs_n_circ": float(np.max(np.abs(layer_refractive_indices[:, 1]))),
        "total_thickness_m": float(np.sum(structure.layer_thicknesses)),
    }


def build_sample_and_aperture(xray_config):
    sample_config = sim.SampleConfig(
        recipe=recipe,
        sample_shape=sample_shape.copy(),
        real_space_pixel_size=real_space_pixel_size,
        xray_config=xray_config,
        sample_name=sample_name,
    )
    sample_config.setup()

    magnetization = pattern_generator.map_magnetization_to_3d(
        magnetic_pattern_x=np.zeros_like(magnetic_pattern_2d),
        magnetic_pattern_y=np.sqrt(np.clip(1 - np.abs(magnetic_pattern_2d) ** 2, 0, 1)),
        magnetic_pattern_z=magnetic_pattern_2d,
        nr_repeats=sample_config.sample_structure.sample_shape[0],
    )
    sample_config.assign_magnetic_pattern(magnetization)

    layer_thicknesses = sample_config.sample_structure.layer_thicknesses
    membrane_index = sample_config.sample_structure.layer_names.index("SiN")
    aperture_taper_depth = float(np.sum(layer_thicknesses[: max(0, membrane_index - 2)]))
    thickness_oh = float(np.sum(layer_thicknesses[:membrane_index]))

    aperture_config = sim.FrontApertureConfig(
        aperture_method="FTH_circular",
        aperture_shape=sample_config.sample_structure.sample_shape,
        real_space_pixel_size=real_space_pixel_size,
        aperture_thicknesses=layer_thicknesses,
        aperture_layer_names=sample_config.sample_structure.layer_names,
        use_roi=True,
        aperture_config={
            "apertures_type": aperture_types,
            "apertures_radius": aperture_radii,
            "apertures_center": aperture_centers,
            "apertures_sigma": aperture_sigmas,
            "apertures_angle": aperture_angles,
            "apertures_ellipticity": aperture_ellipticities,
            "apertures_roughness": aperture_roughnesses,
            "apertures_roughness_modes": aperture_roughness_modes,
            "apertures_seed": aperture_seeds,
            "apertures_top_radius_factor": aperture_top_radius_factors,
            "aperture_taper_depth": aperture_taper_depth,
            "thickness_OH": thickness_oh,
        },
    )
    aperture_config.setup()
    aperture_mask = aperture_config.return_aperture()
    sample_config.assign_aperture_mask(aperture_mask)

    if propagator_method == "Scalar":
        sample_config.sample_structure.calculate_final_scalar_refractive_index(
            pol=pols[0],
            use_aperture_roi=True,
            compact=True,
            lazy=scalar_refractive_index_lazy,
        )
    else:
        contrast_beam_direction = sim.light_beam.beam_direction_from_alpha(illumination_alpha_beam)
        if np.allclose(contrast_beam_direction, [0.0, 0.0, 1.0], atol=1e-14, rtol=0.0):
            contrast_beam_direction = None
        sample_config.sample_structure.calculate_final_dielectric_tensor(
            use_aperture_roi=True,
            compact=True if contrast_beam_direction is None else False,
            beam_direction=contrast_beam_direction,
        )
    return sample_config, aperture_config, aperture_mask


def simulate_one_energy(energy_eV, detector_energy_eV=None, noise_seed_base=1000):
    energy_eV = float(energy_eV)
    detector_energy_eV = energy_eV if detector_energy_eV is None else float(detector_energy_eV)
    xray_config = make_xray_config(energy_eV)
    detector_xray_config = make_xray_config(detector_energy_eV)
    detector_config = make_detector_config(noise_seed=noise_seed_base)
    detector_config.calc_realspace_resolution(detector_xray_config.beam_params)
    sample_config, aperture_config, aperture_mask = build_sample_and_aperture(xray_config)
    material_layers = extract_material_layer_metadata(sample_config, energy_eV)

    illumination_config = sim.IlluminationConfig(
        XRayConfig=xray_config,
        shape=tuple(sample_config.sample_structure.sample_shape[1:]),
        real_space_pixel_size=real_space_pixel_size,
        illumination_function="gaussian",
        illumination_config={
            "center": np.array(illumination_center),
            "distance": illumination_focus_distance,
            "fwhm": illumination_fwhm,
            "alpha_beam": illumination_alpha_beam,
        },
    )
    illumination_config.setup()

    hologram_config = sim.HologramConfig(
        sample_x=sample_config.sample_structure.x,
        sample_y=sample_config.sample_structure.y,
        detector_layout=detector_config.detector_layout,
    )


    for pol_index, pol in enumerate(pols):
        detector_config.detector_params["noise_seed"] = int(noise_seed_base) + pol_index
        illumination_config.update_polarization(pol)
        propagator_config = sim.SamplePropagatorConfig(
            SampleConfig=sample_config,
            IlluminationConfig=illumination_config,
            propagator_method=propagator_method,
            propagator_config={
                "propagator_method": propagator_method,
                "propagate": propagate,
                "jones_apply_zero_order_phase": jones_apply_zero_order_phase,
                "scalar_apply_zero_order_phase": jones_apply_zero_order_phase,
                "scalar_refractive_index_lazy": scalar_refractive_index_lazy,
                "propagation_padding_px": propagation_padding_px,
                "propagation_padding_mode": propagation_padding_mode,
                "propagation_absorber_width_px": propagation_absorber_width_px,
                "propagation_absorber_strength": propagation_absorber_strength,
                "propagation_absorber_profile": propagation_absorber_profile,
                "multislice_propagation_roi": multislice_propagation_roi,
                "multislice_propagation_roi_padding_px": multislice_propagation_roi_padding_px,
                "multislice_propagation_roi_merge_overlaps": multislice_propagation_roi_merge_overlaps,
                "farfield_oversampling": farfield_oversampling,
            },
        )
        propagator_config.setup()
        detector_config.assign_propagated_wavefront(propagator_config)
        detector_config.detect_hologram(
            projection_beam_params=detector_xray_config.beam_params,
        )

        hologram_config.add_exit_waves({pol: propagator_config.return_scalar_wavefield()})
        hologram_config.add_holograms({pol: detector_config.return_ideal_hologram()}, source="ideal")
        hologram_config.add_holograms({pol: detector_config.return_detected_hologram(store_no_beamstop=True)}, source="detected")
        hologram_config.add_holograms({pol: detector_config.return_detected_hologram_without_beamstop()}, source="detected_no_beamstop")

    hologram_config.compute_differences(positive=pols[0], negative=pols[1])
    hologram_config.compute_sums(left=pols[0], right=pols[1])
    fth_hologram_scale = detector_energy_eV / detector_energy_scale_reference()
    if COMPUTE_FTH_RECONSTRUCTIONS:
        compute_energy_aligned_reconstructions(
            hologram_config,
            scale=fth_hologram_scale,
            output_shape=detector_shape,
        )
    else:
        hologram_config.reconstructions = {}
    # Do not compute averaged copies here; the chunk writer saves the original
    # stores directly and avoiding averages prevents another set of large arrays.

    eps = 1e-30
    xmcd_ratio = (hologram_config.exit_waves[pols[0]] + eps) / (hologram_config.exit_waves[pols[1]] + eps)
    xmcd_log = np.log(xmcd_ratio)
    native_reconstruction_pixel_size = detector_config.detector_layout.real_space_resolution
    reconstruction_pixel_size = native_reconstruction_pixel_size * fth_hologram_scale
    supportmask = aperture_config.create_supportmask(
        output_shape=detector_config.detector_layout.detector_shape,
        output_pixel_size=reconstruction_pixel_size,
    )
    recon_oh_slices = fth_object_image_crop_slices(detector_shape, reconstruction_pixel_size, radius_factor=1.0)
    oh_index = aperture_types.index("OH")
    fth_shift_m = fth_reconstruction_shift_sign * (
        np.asarray(aperture_centers[fth_reference_aperture_index], dtype=float)
        - np.asarray(aperture_centers[oh_index], dtype=float)
    )

    return {
        "energy_eV": energy_eV,
        "detector_energy_eV": detector_energy_eV,
        "material_layers": material_layers,
        "supportmask": supportmask,
        "beamstop_mask": np.asarray(detector_config.detector_layout.beamstop).copy(),
        "holograms": hologram_config,
        "xmcd_ratio": xmcd_ratio,
        "xmcd_log": xmcd_log,
        "sample_oh_slices": sample_oh_slices,
        "recon_oh_slices": recon_oh_slices,
        "fth_hologram_scale": fth_hologram_scale,
        "farfield_oversampling": int(farfield_oversampling),
        "reconstruction_pixel_size": reconstruction_pixel_size,
        "fth_reconstruction_shift_m": fth_shift_m,
        "total_ideal_intensity": float(np.sum(hologram_config.ideal_holograms["sum"])),
        "total_detected_counts": float(np.sum(hologram_config.detected_holograms["sum"])),
    }


## 4. Preview The Fixed Aperture Mask

This cell uses the reference energy only to build the material stack. The aperture geometry is then reused at every energy.


In [ ]:

%matplotlib inline
preview_sample, preview_aperture, preview_aperture_mask = build_sample_and_aperture(reference_xray_config)
projection = np.mean(preview_aperture_mask, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(projection, cmap="gray", vmin=0, vmax=1, extent=extent_nm, interpolation="nearest")
axes[0].set_title("front aperture projection")
axes[0].set_xlabel("x (nm)")
axes[0].set_ylabel("y (nm)")
fig.colorbar(im0, ax=axes[0], label="material fraction")

masked_pattern = magnetic_pattern_2d * (projection < 0.5)
im1 = axes[1].imshow(masked_pattern, cmap="RdBu_r", vmin=-1, vmax=1, extent=extent_nm, interpolation="nearest")
axes[1].set_title("magnetic pattern visible through apertures")
axes[1].set_xlabel("x (nm)")
axes[1].set_ylabel("y (nm)")
fig.colorbar(im1, ax=axes[1], label="mz")


del preview_sample, preview_aperture, preview_aperture_mask, projection
gc.collect()


## 5. Run The Energy Sweep With Per-Energy Files

The original notebook did this:

```python
energy_results.append(simulate_one_energy(...))
```

That keeps every large simulation object and every hologram array alive until the
entire sweep finishes. This version instead writes one file per energy in
`SWEEP_OUTPUT_DIR`, deletes the result, and continues. If the kernel dies halfway
through, the completed energy files are still usable.


In [ ]:
SWEEP_OUTPUT_DIR = Path("cobalt_l_edge_energy_sweep_memory_safe_chunks")
OVERWRITE_ENERGY_FILES = True
MANIFEST_PATH = SWEEP_OUTPUT_DIR / "manifest.json"


def write_array(group, name, value):
    """Write a numeric array using compact HDF5-compatible precision."""
    array = np.asarray(value)
    if np.iscomplexobj(array):
        array = array.astype(np.complex64, copy=False)
    elif np.issubdtype(array.dtype, np.floating):
        array = array.astype(np.float32, copy=False)
    group.create_dataset(name, data=array, compression="gzip", shuffle=True)


def write_string_array(group, name, values):
    """Write a one-dimensional UTF-8 string dataset."""
    dtype = h5py.string_dtype(encoding="utf-8")
    group.create_dataset(name, data=np.asarray(values, dtype=dtype))


def write_material_layers(group, metadata):
    """Write layer optical constants used by the propagation setup."""
    group.attrs["energy_eV"] = float(metadata["energy_eV"])
    group.attrs["recipe"] = metadata["recipe"]
    group.attrs["propagator_method"] = metadata["propagator_method"]
    group.attrs["scalar_refractive_index_lazy"] = bool(metadata["scalar_refractive_index_lazy"])
    group.attrs["total_thickness_m"] = float(metadata["total_thickness_m"])
    group.attrs["max_abs_n_circ"] = float(metadata["max_abs_n_circ"])
    write_string_array(group, "layer_names", metadata["layer_names"])
    write_string_array(group, "refractive_index_channel_names", metadata["refractive_index_channel_names"])
    write_array(group, "layer_thicknesses_m", metadata["layer_thicknesses_m"])
    write_array(group, "layer_refractive_indices", metadata["layer_refractive_indices"])
    write_array(group, "effective_refractive_indices_m", metadata["effective_refractive_indices_m"])


def write_energy_result(path, result):
    """Write one simulated energy point and no heavy Python objects."""
    file_mode = "w" if OVERWRITE_ENERGY_FILES else "x"
    holograms = result["holograms"]
    with h5py.File(path, file_mode) as h5:
        h5.attrs["energy_eV"] = float(result["energy_eV"])
        h5.attrs["detector_energy_eV"] = float(result["detector_energy_eV"])
        h5.attrs["fth_hologram_scale"] = float(result["fth_hologram_scale"])
        h5.attrs["farfield_oversampling"] = int(result["farfield_oversampling"])
        h5.attrs["reconstruction_pixel_size"] = float(result["reconstruction_pixel_size"])
        h5.attrs["total_ideal_intensity"] = float(result["total_ideal_intensity"])
        h5.attrs["total_detected_counts"] = float(result["total_detected_counts"])
        h5.attrs["compute_fth_reconstructions"] = bool(COMPUTE_FTH_RECONSTRUCTIONS)
        h5.attrs["pols"] = json.dumps(list(pols))
        write_material_layers(h5.create_group("material_layers"), result["material_layers"])

        output_groups = {
            name: h5.create_group(name)
            for name in (
                "ideal_holograms",
                "detected_holograms",
                "detected_holograms_without_beamstop",
                "supportmask",
                "beamstop_mask",
                "exit_waves",
                "xmcd_log",
                "xmcd_ratio",
            )
        }

        per_energy_stores = {
            "ideal_holograms": holograms.ideal_holograms,
            "detected_holograms": holograms.detected_holograms,
            "detected_holograms_without_beamstop": holograms.detected_holograms_no_beamstop,
            "exit_waves": holograms.exit_waves,
        }
        for group_name, store in per_energy_stores.items():
            for key, value in store.items():
                write_array(output_groups[group_name], key, value)

        write_array(output_groups["supportmask"], "data", result["supportmask"])
        write_array(output_groups["beamstop_mask"], "data", result["beamstop_mask"])
        write_array(output_groups["xmcd_log"], "data", result["xmcd_log"])
        write_array(output_groups["xmcd_ratio"], "data", result["xmcd_ratio"])

        if COMPUTE_FTH_RECONSTRUCTIONS and getattr(holograms, "reconstructions", None):
            recon_group = h5.create_group("reconstructions")
            for source, store in holograms.reconstructions.items():
                source_group = recon_group.create_group(source)
                for key, value in store.items():
                    write_array(source_group, key, value)


def write_manifest(entries):
    SWEEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(entries, indent=2))


energy_manifest = []

if RUN_ENERGY_SWEEP:
    SWEEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for i, energy in enumerate(energy_values_eV):
        energy = float(energy)
        energy_key = f"{i:05d}"
        output_path = SWEEP_OUTPUT_DIR / f"energy_{energy_key}_{energy:.2f}eV.h5"
        detector_energy = detector_energy_for_index(i, energy)
        if np.isclose(detector_energy, energy):
            energy_label = f"{energy:.2f} eV"
        else:
            energy_label = f"{energy:.2f} eV, detector {detector_energy:.2f} eV"
        print(f"[{i + 1:02d}/{len(energy_values_eV)}] Simulating {energy_label} -> {output_path.name}", flush=True)

        result = simulate_one_energy(energy, detector_energy_eV=detector_energy, noise_seed_base=1000 + 10 * i)
        write_energy_result(output_path, result)

        entry = {
            "index": i,
            "energy_eV": energy,
            "detector_energy_eV": detector_energy,
            "file": output_path.name,
            "total_ideal_intensity": float(result["total_ideal_intensity"]),
            "total_detected_counts": float(result["total_detected_counts"]),
            "farfield_oversampling": int(result["farfield_oversampling"]),
            "max_abs_n_circ": float(result["material_layers"]["max_abs_n_circ"]),
            "material_layers_group": "material_layers",
        }
        energy_manifest.append(entry)
        write_manifest(energy_manifest)

        del result
        gc.collect()
        plt.close("all")
        print(f"  wrote {output_path.name}; kept only manifest entry in RAM", flush=True)

    print(f"Done. Wrote {len(energy_manifest)} energy files to {SWEEP_OUTPUT_DIR.resolve()}")
    print(f"Manifest: {MANIFEST_PATH.resolve()}")
else:
    print("Sweep disabled. Set RUN_ENERGY_SWEEP = True and rerun this cell.")


## 6. Inspect Completed Files

This cell checks what was written without loading all holograms into memory.


In [ ]:
if MANIFEST_PATH.exists():
    completed = json.loads(MANIFEST_PATH.read_text())
    print(f"Completed energies: {len(completed)}")
    if completed:
        print("First:", completed[0])
        print("Last:", completed[-1])
else:
    print("No manifest found yet.")